In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
create schema if not exists retailanalytics.silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retailanalytics.silver.products
(
    ProductID INT,
    ProductName STRING,
    Category STRING,
    SubCategory STRING,
    Brand STRING,
    CostPrice DECIMAL(10,2),

    _AdfPipelineRunId STRING,
    _ProcessedTimestamp TIMESTAMP,
    _IsRejected BOOLEAN
)
USING DELTA;

CREATE TABLE IF NOT EXISTS retailanalytics.silver.customers
(
    CustomerID INT,
    FirstName STRING,
    LastName STRING,
    Email STRING,
    Phone STRING,
    City STRING,
    State STRING,
    LastUpdated TIMESTAMP,

    _AdfPipelineRunId STRING,
    _ProcessedTimestamp TIMESTAMP,
    _IsRejected BOOLEAN
)
USING DELTA;

CREATE TABLE IF NOT EXISTS retailanalytics.silver.orders
(
    OrderID BIGINT,
    CustomerID INT,
    ProductID INT,
    OrderDate DATE,
    Quantity INT,
    UnitPrice DECIMAL(10,2),
    StoreCode STRING,

    _AdfPipelineRunId STRING,
    _ProcessedTimestamp TIMESTAMP,
    _IsRejected BOOLEAN
)
USING DELTA;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS retailanalytics.reject.products_reject
(
    ProductID STRING,
    ProductName STRING,
    Category STRING,
    SubCategory STRING,
    Brand STRING,
    CostPrice STRING,
    RejectReason STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS retailanalytics.reject.customers_reject
(
    CustomerID STRING,
    FirstName STRING,
    LastName STRING,
    Email STRING,
    Phone STRING,
    City STRING,
    State STRING,
    LastUpdated STRING,
    RejectReason STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS retailanalytics.reject.orders_reject
(
    OrderID STRING,
    CustomerID STRING,
    ProductID STRING,
    OrderDate STRING,
    Quantity STRING,
    UnitPrice STRING,
    StoreCode STRING,
    RejectReason STRING
)
USING DELTA;

In [0]:
config_df = spark.table("retailanalytics.config.table_config")

silver_configs = [
    row.TableName
    for row in (config_df.filter(col("ActiveFlag") == "Y")
        .filter(col("TableName") != "exchange_rates").collect())
]

audit_data = []

In [0]:
for table in silver_configs:
    source_df = spark.table(f"retailanalytics.bronze.{table}")
    valid_df = source_df
    reject_df = spark.createDataFrame([], source_df.schema)

    if table == "products":
        valid_df = source_df.filter(col("ProductID").isNotNull())
        reject_df = source_df.filter(col("ProductID").isNull())
        df = (valid_df.withColumn("ProductID", col("ProductID").cast("int"))
            .withColumn("CostPrice", col("CostPrice").cast("decimal(10,2)")))
        
    elif table == "customers":
        valid_df = source_df.filter(col("CustomerID").isNotNull())
        reject_df = source_df.filter(col("CustomerID").isNull())
        df = (valid_df.withColumn("CustomerID", col("CustomerID").cast("int"))
            .withColumn("LastUpdated", to_timestamp("LastUpdated")))

    elif table == "orders":
        valid_df = source_df.filter(col("CustomerID").isNotNull() &
        col("ProductID").isNotNull() & (col("Quantity").cast("int") > 0) &
        (col("UnitPrice").cast("double") > 0))

        reject_df = source_df.filter(col("CustomerID").isNull() | col("ProductID").isNull()
        | (col("Quantity").cast("int") <= 0) | (col("UnitPrice").cast("double") <= 0))

        df = (valid_df.withColumn("OrderID", col("OrderID").cast("bigint"))
            .withColumn("CustomerID", col("CustomerID").cast("int"))
            .withColumn("ProductID", col("ProductID").cast("int"))
            .withColumn("OrderDate", to_date("OrderDate"))
            .withColumn("Quantity", col("Quantity").cast("int"))
            .withColumn("UnitPrice", col("UnitPrice").cast("decimal(10,2)")))

    df = (df.drop("_IngestionTimestamp")
        .withColumn("_ProcessedTimestamp", current_timestamp())
        .withColumn("_IsRejected", lit(False)))

    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"retailanalytics.silver.{table}"))

    reject_df = reject_df.withColumn("RejectReason",lit("Data Quality Failure"))

    (reject_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"retailanalytics.reject.{table}_reject"))

    audit_data.append((table,source_df.count(),valid_df.count(),reject_df.count()))

    print(f"Silver Loaded : {table}")

Silver Loaded : products
Silver Loaded : customers
Silver Loaded : orders


In [0]:
silver_tables = ["products","customers","orders"]

for table in silver_tables:
    print("\n")
    print("=" * 50)
    print(table.upper())
    print("=" * 50)
    display(spark.table(f"retailanalytics.silver.{table}").limit(5))
    print("Count :",spark.table(f"retailanalytics.silver.{table}").count())



PRODUCTS


ProductID,ProductName,Category,SubCategory,Brand,CostPrice,_AdfPipelineRunId,_ProcessedTimestamp,_IsRejected
1,Laptop_1,Electronics,Laptop,Sony,252.44,BRONZE_LOAD_001,2026-06-25T05:52:14.187Z,false
2,Jacket_2,Fashion,Jacket,Samsung,679.93,BRONZE_LOAD_001,2026-06-25T05:52:14.187Z,false
3,Tablet_3,Electronics,Tablet,Nike,41.46,BRONZE_LOAD_001,2026-06-25T05:52:14.187Z,false
4,Laptop_4,Electronics,Laptop,Apple,510.30,BRONZE_LOAD_001,2026-06-25T05:52:14.187Z,false
5,Tablet_5,Electronics,Tablet,Apple,718.86,BRONZE_LOAD_001,2026-06-25T05:52:14.187Z,false


Count : 62


CUSTOMERS


CustomerID,FirstName,LastName,Email,Phone,City,State,LastUpdated,_AdfPipelineRunId,_ProcessedTimestamp,_IsRejected
1,Arjun,Patel,customer1@gmail.com,9927622808,Mumbai,Maharashtra,2026-06-25T05:26:26.033Z,BRONZE_LOAD_001,2026-06-25T05:52:18.173Z,false
2,Arjun,Iyer,customer2@gmail.com,9885934963,Mumbai,Maharashtra,2026-06-25T05:26:26.033Z,BRONZE_LOAD_001,2026-06-25T05:52:18.173Z,false
3,Anjali,Patel,customer3@gmail.com,9959015427,Delhi,Delhi,2026-06-25T05:26:26.033Z,BRONZE_LOAD_001,2026-06-25T05:52:18.173Z,false
4,Rahul,Singh,customer4@gmail.com,9186064385,Chennai,Tamil Nadu,2026-06-25T05:26:26.033Z,BRONZE_LOAD_001,2026-06-25T05:52:18.173Z,false
5,Divya,Reddy,customer5@gmail.com,9807398273,Chennai,Tamil Nadu,2026-06-25T05:26:26.033Z,BRONZE_LOAD_001,2026-06-25T05:52:18.173Z,false


Count : 1000


ORDERS


OrderID,CustomerID,ProductID,OrderDate,Quantity,UnitPrice,StoreCode,_AdfPipelineRunId,_ProcessedTimestamp,_IsRejected
1,924,105,2025-01-12,1,757.10,HYD001,BRONZE_LOAD_001,2026-06-25T05:52:22.012Z,false
2,709,226,2025-01-03,2,3607.22,MUM001,BRONZE_LOAD_001,2026-06-25T05:52:22.012Z,false
3,450,236,2025-03-13,2,1004.64,CHN001,BRONZE_LOAD_001,2026-06-25T05:52:22.012Z,false
4,722,359,2025-03-18,4,1787.20,CHN001,BRONZE_LOAD_001,2026-06-25T05:52:22.012Z,false
5,446,118,2025-04-29,4,1507.35,CHN001,BRONZE_LOAD_001,2026-06-25T05:52:22.012Z,false


Count : 5000



# EXCHANGE RATES SILVER


In [0]:
spark.table("retailanalytics.bronze.exchange_rates").show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+--------------------------+
|raw_json                                                                                                                                        |_AdfPipelineRunId|_IngestionTimestamp       |
+------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+--------------------------+
|{    "base": "USD",    "date": "2026-06-25",    "rates": {        "INR": 86.54,        "EUR": 0.91,        "GBP": 0.78,        "AED": 3.67    }}|BRONZE_LOAD_001  |2026-06-25 05:47:56.242862|
+------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+--------------------------+



In [0]:
%sql

CREATE TABLE IF NOT EXISTS retailanalytics.silver.exchange_rates
(
    BaseCurrency STRING,
    TargetCurrency STRING,
    ExchangeRate DECIMAL(12,6),
    RateDate DATE,

    _AdfPipelineRunId STRING,
    _ProcessedTimestamp TIMESTAMP,
    _IsRejected BOOLEAN
)
USING DELTA;

In [0]:
bronze_exchange = spark.table("retailanalytics.bronze.exchange_rates")

display(bronze_exchange)

raw_json,_AdfPipelineRunId,_IngestionTimestamp
"{ ""base"": ""USD"", ""date"": ""2026-06-25"", ""rates"": { ""INR"": 86.54, ""EUR"": 0.91, ""GBP"": 0.78, ""AED"": 3.67 }}",BRONZE_LOAD_001,2026-06-25T05:47:56.242Z


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

rates_schema = MapType(StringType(),DoubleType())
parsed_df = (bronze_exchange.withColumn("rates_map",from_json(to_json(col("rates")),rates_schema)))

display(parsed_df)

In [0]:
exchange_silver = (parsed_df.select(col("base_code").alias("BaseCurrency"),
                                    current_date().alias("RateDate"),"_AdfPipelineRunId",
        explode(col("rates_map")).alias("TargetCurrency","ExchangeRate"))
    .withColumn("_ProcessedTimestamp",current_timestamp())
    .withColumn("_IsRejected",lit(False))
)

display(exchange_silver)

raw_json,_AdfPipelineRunId,_IngestionTimestamp,json_data
"{ ""base"": ""USD"", ""date"": ""2026-06-25"", ""rates"": { ""INR"": 86.54, ""EUR"": 0.91, ""GBP"": 0.78, ""AED"": 3.67 }}",BRONZE_LOAD_001,2026-06-25T05:47:56.242Z,"List(USD, 2026-06-25, List(86.54, 0.91, 0.78, 3.67))"


In [0]:
(exchange_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
.saveAsTable("retailanalytics.silver.exchange_rates"))

print("Exchange Rates Silver Loaded")

Exchange Rates Silver Loaded


In [0]:
display(spark.table("retailanalytics.silver.exchange_rates"))

BaseCurrency,RateDate,_AdfPipelineRunId,TargetCurrency,ExchangeRate,_ProcessedTimestamp,_IsRejected
USD,2026-06-25,BRONZE_LOAD_001,INR,86.54,2026-06-25T05:52:44.881Z,false
USD,2026-06-25,BRONZE_LOAD_001,EUR,0.91,2026-06-25T05:52:44.881Z,false
USD,2026-06-25,BRONZE_LOAD_001,GBP,0.78,2026-06-25T05:52:44.881Z,false
USD,2026-06-25,BRONZE_LOAD_001,AED,3.67,2026-06-25T05:52:44.881Z,false


In [0]:
audit_df = spark.createDataFrame(audit_data,["TableName","SourceCount","TargetCount","RejectCount"])
audit_df = audit_df.withColumn("LoadTimestamp",current_timestamp())
(audit_df.write.format("delta").mode("append").saveAsTable("retailanalytics.audit.pipeline_audit"))

In [0]:
display(spark.table("retailanalytics.audit.pipeline_audit"))
display(spark.table("retailanalytics.reject.orders_reject"))

PipelineRunId,TableName,SourceCount,TargetCount,RejectCount,LoadTimestamp,Status
null,products,62,62,0,2026-06-25T05:52:48.234Z,null
null,customers,1000,1000,0,2026-06-25T05:52:48.234Z,null
null,orders,5000,5000,0,2026-06-25T05:52:48.234Z,null


OrderID,CustomerID,ProductID,OrderDate,Quantity,UnitPrice,StoreCode,_AdfPipelineRunId,_IngestionTimestamp,RejectReason


In [0]:
display(spark.table("retailanalytics.audit.pipeline_audit"))

print(spark.table("retailanalytics.reject.orders_reject").count())
print(spark.table("retailanalytics.reject.products_reject").count())
print(spark.table("retailanalytics.reject.customers_reject").count())

PipelineRunId,TableName,SourceCount,TargetCount,RejectCount,LoadTimestamp,Status
null,products,62,62,0,2026-06-25T05:52:48.234Z,null
null,customers,1000,1000,0,2026-06-25T05:52:48.234Z,null
null,orders,5000,5000,0,2026-06-25T05:52:48.234Z,null


0
0
0
